In [78]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import Subset

In [79]:
transform = transforms.Compose([
    transforms.ToTensor(),  
])

In [80]:
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

In [14]:
batch_size = 32
num_teachers = 100

In [15]:
def teacher_load(train_data, num_teachers):
  teacher_loaders = []
  data_size = len(train_dataset) // num_teachers
  for i in range(num_teachers):
    indices = list(range(i*data_size, (i+1)*data_size))
    subset = Subset(train_dataset, indices)
    loader = torch.utils.data.DataLoader(subset, batch_size=batch_size)
    teacher_loaders.append(loader)
  return teacher_loaders
teacher_loader = teacher_load(train_dataset, num_teachers)

In [16]:
num_student_train_set = 9000
student_train_data = Subset(test_dataset, list(range(num_student_train_set)))
student_test_data = Subset(test_dataset, list(range(num_student_train_set, len(test_dataset))))
student_train_loader = torch.utils.data.DataLoader(student_train_data, batch_size=batch_size)
student_test_loader = torch.utils.data.DataLoader(student_test_data, batch_size=batch_size)

In [64]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(120, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 120)  # Flatten the tensor
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [66]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
epochs=20

In [67]:
def train(model, trainloader, criterion, optimizer, epochs):
  model.to(device)
  running_loss = 0
  for e in range(epochs):
    model.train()
    for images, labels in trainloader:
      images, labels = images.to(device), labels.to(device)
 
      optimizer.zero_grad()
 
      output = model(images)
      loss = criterion(output, labels)
      loss.backward()
      optimizer.step()
 
      running_loss += loss.item()

In [68]:
def train_teachers(num_teachers, teacher_loaders):
  models = []
  for t in range(num_teachers):
    print("Training teacher {}".format(t+1))
    model = CNN()
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.003)
    train(model, teacher_loaders[t], criterion, optimizer, epochs)
    models.append(model)
  return models

In [69]:
teacher_models = train_teachers(num_teachers, teacher_loader)

Training teacher 1
Training teacher 2
Training teacher 3
Training teacher 4
Training teacher 5
Training teacher 6
Training teacher 7
Training teacher 8
Training teacher 9
Training teacher 10
Training teacher 11
Training teacher 12
Training teacher 13
Training teacher 14
Training teacher 15
Training teacher 16
Training teacher 17
Training teacher 18
Training teacher 19
Training teacher 20
Training teacher 21
Training teacher 22
Training teacher 23
Training teacher 24
Training teacher 25
Training teacher 26
Training teacher 27
Training teacher 28
Training teacher 29
Training teacher 30
Training teacher 31
Training teacher 32
Training teacher 33
Training teacher 34
Training teacher 35
Training teacher 36
Training teacher 37
Training teacher 38
Training teacher 39
Training teacher 40
Training teacher 41
Training teacher 42
Training teacher 43
Training teacher 44
Training teacher 45
Training teacher 46
Training teacher 47
Training teacher 48
Training teacher 49
Training teacher 50
Training 

In [70]:
def predict(model, dataloader):
  outputs = torch.zeros(0, dtype=torch.long).to(device)
  model.to(device)
  model.eval()
  for images, labels in dataloader:
    images, labels = images.to(device), labels.to(device)
    output = model(images)
    ps = torch.argmax(torch.exp(output), dim=1)
    outputs = torch.cat((outputs, ps))
 
  return outputs

In [84]:
def prediction_fn(models, data_loader):
  preds = torch.torch.zeros((len(models), num_student_train_set), dtype=torch.long)
  for i, model in enumerate(models):
    results = predict(model, data_loader)
    preds[i] = results
  return preds.numpy()

In [72]:
preds = prediction_fn(teacher_models, student_train_loader)

In [85]:
def evaluate_teacher(model, loader):
    correct, total = 0, 0
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [86]:
print("Teacher 1 accuracy:", evaluate_teacher(teacher_models[0], student_train_loader))

Teacher 1 accuracy: 0.8835555555555555


In [87]:
def get_student_labels(preds, epsilon):
    labels = []
    beta = 1 / epsilon
    for image_preds in np.transpose(preds):
        label_counts = np.bincount(image_preds, minlength=10).astype(float)
        noisy_counts = label_counts + np.random.laplace(0, beta, size=10)
        new_label = np.argmax(noisy_counts)
        labels.append(new_label)
    return np.array(labels, dtype=int)

In [88]:
epsilon = 5
student_labels = get_student_labels(preds, epsilon)

In [89]:
def student_loader(student_train_loader, labels):
  student_iterator = iter(student_train_loader)
  for i, (data, _) in enumerate(student_iterator):
    student_train_label = torch.from_numpy(labels[i*len(data):(i+1)*len(data)])
    yield data, student_train_label

In [92]:
student_model = CNN()
criterion = nn.NLLLoss()
optimizer = optim.Adam(student_model.parameters(), lr=0.001)
student_model.to(device)
epochs=10
steps = 0
running_loss = 0

In [93]:
for e in range(epochs):
    student_model.train()
    train_loader = student_loader(student_train_loader, student_labels)
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        steps += 1

        optimizer.zero_grad()
        output = student_model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if steps % 50 == 0:
            test_loss = 0
            accuracy = 0
            student_model.eval()

            with torch.no_grad():
                for images, labels in student_test_loader:
                    images, labels = images.to(device), labels.to(device)
                    log_ps = student_model(images)
                    test_loss += criterion(log_ps, labels).item()

                    top_p, top_class = log_ps.topk(1, dim=1)
                    equals = top_class == labels.view(*top_class.shape)
                    accuracy += torch.mean(equals.type(torch.FloatTensor))

            student_model.train()
            print("Epoch: {}/{}.. ".format(e+1, epochs),
                  "Training Loss: {:.3f}.. ".format(running_loss/len(student_train_loader)),
                  "Test Loss: {:.3f}.. ".format(test_loss/len(student_test_loader)),
                  "Test Accuracy: {:.3f}".format(accuracy/len(student_test_loader)))
            running_loss = 0

Epoch: 1/10..  Training Loss: 0.272..  Test Loss: 0.962..  Test Accuracy: 0.731
Epoch: 1/10..  Training Loss: 0.097..  Test Loss: 0.547..  Test Accuracy: 0.846
Epoch: 1/10..  Training Loss: 0.058..  Test Loss: 0.492..  Test Accuracy: 0.856
Epoch: 1/10..  Training Loss: 0.036..  Test Loss: 0.415..  Test Accuracy: 0.883
Epoch: 1/10..  Training Loss: 0.035..  Test Loss: 0.348..  Test Accuracy: 0.905
Epoch: 2/10..  Training Loss: 0.046..  Test Loss: 0.420..  Test Accuracy: 0.872
Epoch: 2/10..  Training Loss: 0.031..  Test Loss: 0.323..  Test Accuracy: 0.914
Epoch: 2/10..  Training Loss: 0.024..  Test Loss: 0.336..  Test Accuracy: 0.910
Epoch: 2/10..  Training Loss: 0.022..  Test Loss: 0.328..  Test Accuracy: 0.910
Epoch: 2/10..  Training Loss: 0.016..  Test Loss: 0.371..  Test Accuracy: 0.898
Epoch: 2/10..  Training Loss: 0.015..  Test Loss: 0.317..  Test Accuracy: 0.914
Epoch: 3/10..  Training Loss: 0.040..  Test Loss: 0.323..  Test Accuracy: 0.910
Epoch: 3/10..  Training Loss: 0.018..  T